# CXR-Recall — Phase 2 text-variant study: Findings-only (and Impression-only) report text

Separate notebook from `cxr_recall_phase2_kaggle.ipynb`. It tests the one lever proposed against the **circularity** problem found in Phase 2: the converged text-only probe recovers the 14 MeSH labels at **0.9355 test AUROC** while the image alone gets **0.6944**, and that gap does not shrink on reports whose text never appears in training (RESULTS.md, "Phase 2 rerun at batch 64"). The labels are NLM's own indexing of these reports, so a text embedding already contains them. The proposed lever: drop the **Impression** (assumed to be the section that most directly restates the diagnosis) and use the **Findings** alone.

**A pre-check run before building this notebook says the premise is probably wrong for this dataset.** Word-level TF-IDF + logistic regression on the real `SEED=42` split recovers the labels from **Findings alone at 0.9767 test AUROC** (0.9892 on rows that have Findings), from the **Impression alone at only 0.8347**, and from both at 0.9930. The Impression is short (median 5 words) and generic; the Findings hold the detailed descriptions the MeSH tags were assigned from. So this notebook is built to *measure*, not assume, and puts the cheap checks first:

| Section | What it answers | Cost |
|---|---|---|
| 4. Lexical ceiling | How much label signal sits in each report section, from words alone (TF-IDF + logistic regression)? | seconds, CPU |
| 5. Raw ClinicalBERT probes | Same question through a frozen, non-contrastive Bio_ClinicalBERT, Findings vs Impression vs both | ~5 min, GPU |
| 6-9. Stage 1 + probes | The full Phase 2 pipeline with `TEXT_VARIANT` as the report text: contrastive pretraining, cached embeddings, **converged** probes, subsets, Arm 2 | ~45 min, GPU |

**Design decisions (all visible in the code).**
- **13.4% of rows (997 of 7,466) have no Findings.** They are spread evenly across normal and abnormal studies (13.0% vs 13.6%), so missingness does not bias the classes. They are kept in the same train/val/test split (so numbers stay comparable with every earlier run), get the empty string as text, are **excluded from Stage 1 contrastive pairs** (an empty report has nothing to align with and gives identical embeddings that act as false negatives), and every result is reported on all rows *and* on the rows that have text.
- **Stage 1 differs from earlier runs in two ways:** same-study pairs are masked out of the contrastive negatives (`MASK_SAME_STUDY`, the audit fix: a study's frontal and lateral views share one report) and empty-text pairs are dropped. So a comparison with the earlier full-text numbers is confounded by both changes and by Stage 1's run-to-run noise (about 0.03 on retrieval, about 0.01 on probes). The clean comparisons are inside this notebook: Section 5 (same encoder, three text variants), and a control rerun with `TEXT_VARIANT = 'full'` (same Stage 1 recipe, full text).
- **Probes are trained to convergence** (sklearn L-BFGS logistic regression, C chosen on validation; the gated head in torch at lr 1e-2 with a stop-on-cap warning). Earlier Phase 2 probes were under-trained; see RESULTS.md.
- Labels are built exactly as in every earlier run (including the known comma-splitting quirk in `build_tags`) so the 14-label vocabulary and the split are identical.

**What to expect.** If Findings-only text-only still scores well above 0.9, the conclusion is that report-derived MeSH labels cannot test whether text helps, whichever section is used. Arm 3 (evidence fusion) is deliberately not rebuilt here; it is a Conclusion-1 experiment and does not depend on the text variant.

Attach the same dataset (+ Add Data → "Chest X-rays Indiana University"), turn on **GPU** and **Internet** (`Bio_ClinicalBERT` downloads from the Hub). Use **Save Version → Save & Run All**.

In [ ]:
import torch
import random
import numpy as np

print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print('seed set to', SEED)

In [ ]:
# ---------------- configuration (edit here) ----------------
TEXT_VARIANT = 'findings'   # report text fed to Stage 1 and the probes: 'findings' | 'impression' | 'full'
                            # 'full' = Findings + Impression, the text every earlier Phase 2 run used (use it for the control run)
RUN_STAGE1 = True           # False = run only Sections 4-5 (cheap diagnostics, ~10 min) and stop
STAGE1_BATCH_SIZE = 64      # 64 + mixed precision fit on a Kaggle GPU in earlier runs; drop to 32 on an OOM
MASK_SAME_STUDY = True      # exclude same-study off-diagonal pairs from the contrastive negatives (audit fix)
assert TEXT_VARIANT in ('findings', 'impression', 'full'), TEXT_VARIANT
print('TEXT_VARIANT =', TEXT_VARIANT, '| RUN_STAGE1 =', RUN_STAGE1, '| STAGE1_BATCH_SIZE =', STAGE1_BATCH_SIZE, '| MASK_SAME_STUDY =', MASK_SAME_STUDY)

## 1. Locate the attached dataset

In [ ]:
import os

INPUT_DIR = '/kaggle/input'
candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached -- use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 2. Build the manifest, keeping Findings and Impression separate

In [ ]:
import pandas as pd

reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','   # identical to every earlier run (known quirk kept so the labels and split stay comparable)
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

findings = merged[FINDINGS_COL].fillna('').astype(str).str.strip()
impression = merged[IMPRESSION_COL].fillna('').astype(str).str.strip()
manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'mesh_tags': merged.apply(build_tags, axis=1),
    'findings': findings,
    'impression': impression,
    # 'full' reproduces the report_text every earlier Phase 2 run used (non-empty sections joined by a space)
    'full': (findings + ' ' + impression).str.strip(),
})
manifest['has_findings'] = manifest['findings'] != ''
manifest['report_text'] = manifest[TEXT_VARIANT]            # the text variant fed to Stage 1 and the probes
manifest['has_text'] = manifest['report_text'] != ''

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
print(f"rows without Findings: {(~manifest['has_findings']).sum()} ({(~manifest['has_findings']).mean():.1%}) | "
      f"rows without text for TEXT_VARIANT={TEXT_VARIANT!r}: {(~manifest['has_text']).sum()} ({(~manifest['has_text']).mean():.1%})")
print('median words  findings:', int(manifest.loc[manifest.has_findings, 'findings'].str.split().str.len().median()),
      ' impression:', int(manifest.loc[manifest.impression != '', 'impression'].str.split().str.len().median()))
manifest.head(2)

## 3. Label vocabulary + patient-level split (identical to every earlier run, `seed=42`)

In [ ]:
from collections import Counter

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

In [ ]:
import copy
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import roc_auc_score

transform = ViT_B_16_Weights.DEFAULT.transforms()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, LR = 16, 1e-4
EPOCHS_REAL, PATIENCE = 20, 5
criterion = nn.BCEWithLogitsLoss()

def macro_auroc(y_true, y_score):
    scores = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        scores.append(roc_auc_score(y_true[:, i], y_score[:, i]))
    return float(np.mean(scores)) if scores else float('nan')

def held_out_auroc(forward_fn, loader, label_pos):
    """One pass over a held-out loader with the model already in eval() mode.
    forward_fn(batch_tuple) -> logits; label_pos = index of the label tensor in
    the batch tuple. Used for TEST numbers: val is what every arm's early
    stopping selects on, so val AUROC is an optimistically-selected number;
    test is evaluated exactly once per arm, on the reloaded best checkpoint,
    and never influences any training or selection decision."""
    all_labels, all_scores = [], []
    with torch.no_grad():
        for batch in loader:
            all_scores.append(torch.sigmoid(forward_fn(batch)).cpu().numpy())
            all_labels.append(batch[label_pos].numpy())
    return macro_auroc(np.concatenate(all_labels), np.concatenate(all_scores))

print('setup complete -- device:', device, ' batch size:', BATCH_SIZE, ' lr:', LR)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss

def build_index(embeddings):
    embeddings = embeddings.astype(np.float32).copy()
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def search(index, queries, k):
    queries = queries.astype(np.float32).copy()
    faiss.normalize_L2(queries)
    return index.search(queries, k)

def label_agreement_at_k(query_labels, retrieved_labels):
    if retrieved_labels.shape[0] == 0:
        return float('nan')
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    return float(shares.mean())

def mean_reciprocal_rank(query_labels, retrieved_labels):
    shares = (retrieved_labels * query_labels).sum(axis=1) > 0
    hits = np.nonzero(shares)[0]
    return 0.0 if len(hits) == 0 else 1.0 / (hits[0] + 1)

K = 5
print('retrieval utilities defined -- k =', K)

In [ ]:
# Shared helpers for the linear probes used in Sections 4, 5 and 8.
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def labels_of(df):
    return np.stack([mesh_tags_to_vector(t.split(';') if t else [], label_vocab) for t in df['mesh_tags']])

y_tr, y_va, y_te = labels_of(train_df), labels_of(val_df), labels_of(test_df)
hasF_va, hasF_te = val_df['has_findings'].to_numpy(), test_df['has_findings'].to_numpy()

def masked_auroc(y, p, mask=None):
    if mask is not None:
        y, p = y[mask], p[mask]
    return macro_auroc(y, p)

def per_label_auroc(y, p):
    return [roc_auc_score(y[:, i], p[:, i]) if len(np.unique(y[:, i])) >= 2 else float('nan') for i in range(y.shape[1])]

def fit_linear_probe(Xtr, ytr, Xva, yva, Xte, Cs):
    """One logistic regression per label (sklearn L-BFGS -- a convex problem solved to tolerance, so 'converged' is
    a fact, not a hope), a single C for all labels chosen by validation macro AUROC. Returns (C, val_scores, test_scores)."""
    best = None
    for C in Cs:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            clfs = [LogisticRegression(C=C, max_iter=5000).fit(Xtr, ytr[:, j]) for j in range(ytr.shape[1])]
        pv = np.stack([c.decision_function(Xva) for c in clfs], 1)
        v = macro_auroc(yva, pv)
        if best is None or v > best[0]:
            best = (v, C, clfs, pv)
    v, C, clfs, pv = best
    return C, pv, np.stack([c.decision_function(Xte) for c in clfs], 1)

print('probe helpers defined')

## 4. Lexical ceiling: how much label signal is in each report section? (words only, no neural network)

TF-IDF (unigrams + bigrams, fit on train only) + one logistic regression per label. This is the ceiling for what *any* text model could recover by matching words, and it is the cleanest measure of circularity per section. Reported on all rows and on the rows that have Findings (so Findings / Impression / both are compared on the same rows). Pre-check on the real split, run locally: Findings 0.9767, Impression 0.8347, both 0.9930 (test, all rows).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

LOCAL_PRECHECK = {'findings': 0.9767, 'impression': 0.8347, 'full': 0.9930}   # test, all rows, run locally before building this notebook
lexical = {}
for variant in ('findings', 'impression', 'full'):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Xtr, Xva, Xte = vec.fit_transform(train_df[variant]), vec.transform(val_df[variant]), vec.transform(test_df[variant])
    C, pv, pt = fit_linear_probe(Xtr, y_tr, Xva, y_va, Xte, Cs=(0.3, 1, 3, 10, 30, 100))
    lexical[variant] = {'C': C, 'val': masked_auroc(y_va, pv), 'test': masked_auroc(y_te, pt),
                        'val_hasF': masked_auroc(y_va, pv, hasF_va), 'test_hasF': masked_auroc(y_te, pt, hasF_te),
                        'per_label': per_label_auroc(y_te, pt)}

print('=== word-level label recovery (TF-IDF + logistic regression), macro AUROC ===')
print(f"{'report text':14s} {'val':>7s} {'TEST':>7s} {'TEST, rows with Findings':>26s}   local pre-check")
for variant, name in (('findings', 'Findings only'), ('impression', 'Impression only'), ('full', 'both')):
    r = lexical[variant]
    print(f"{name:14s} {r['val']:7.4f} {r['test']:7.4f} {r['test_hasF']:26.4f}   {LOCAL_PRECHECK[variant]:.4f}")
print()
print(f"{'label':24s} {'findings':>9s} {'impression':>11s} {'both':>7s}   (test AUROC per label)")
for k, v in enumerate(label_vocab):
    print(f"{v:24s} {lexical['findings']['per_label'][k]:9.3f} {lexical['impression']['per_label'][k]:11.3f} {lexical['full']['per_label'][k]:7.3f}")
print()
drop = lexical['full']['test'] - lexical['findings']['test']
print(f"Dropping the Impression lowers word-level label recovery by only {drop:.4f} AUROC "
      f"({lexical['full']['test']:.4f} -> {lexical['findings']['test']:.4f}); the Impression alone reaches {lexical['impression']['test']:.4f}.")
if lexical['findings']['test'] > 0.95:
    print('=> The label signal lives in the Findings, not the Impression. Findings-only text cannot be expected to remove the circularity.')
else:
    print('=> Findings-only text carries materially less label signal than the full report; it may weaken the shortcut.')

## 5. The same question through a frozen, non-contrastive Bio_ClinicalBERT

Mean-free `[CLS]` embeddings (768-d, standardised) from the pretrained encoder with **no Stage 1 training**, then converged logistic-regression probes. This is the controlled same-encoder comparison of the three text variants (Findings only, Impression only, both) that Stage 1 runs cannot give, because each Stage 1 run has its own noise.

In [ ]:
from transformers import AutoModel, AutoTokenizer

_bert_name = 'emilyalsentzer/Bio_ClinicalBERT'
_bert_tok = AutoTokenizer.from_pretrained(_bert_name)
_bert = AutoModel.from_pretrained(_bert_name).to(device).eval()

@torch.no_grad()
def bert_cls(texts, bs=64, max_length=192):
    out = []
    for s in range(0, len(texts), bs):
        tokens = _bert_tok(texts[s:s + bs], padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
        with torch.autocast(device.type, enabled=(device.type == 'cuda')):
            h = _bert(**tokens).last_hidden_state[:, 0, :]
        out.append(h.float().cpu().numpy())
    return np.concatenate(out)

print('Bio_ClinicalBERT loaded for raw-embedding probes')

In [ ]:
from sklearn.preprocessing import StandardScaler

raw_res = {}
for variant in ('findings', 'impression', 'full'):
    emb = {sp: bert_cls(df[variant].tolist()) for sp, df in (('train', train_df), ('val', val_df), ('test', test_df))}
    sc = StandardScaler().fit(emb['train'])
    Xtr, Xva, Xte = (sc.transform(emb[s]) for s in ('train', 'val', 'test'))
    C, pv, pt = fit_linear_probe(Xtr, y_tr, Xva, y_va, Xte, Cs=(0.001, 0.003, 0.01, 0.03, 0.1, 0.3))
    raw_res[variant] = {'C': C, 'val': masked_auroc(y_va, pv), 'test': masked_auroc(y_te, pt), 'test_hasF': masked_auroc(y_te, pt, hasF_te)}

print('=== frozen pretrained Bio_ClinicalBERT [CLS] + converged linear probe, macro AUROC ===')
print(f"{'report text':14s} {'val':>7s} {'TEST':>7s} {'TEST, rows with Findings':>26s}   (word-level ceiling, test)")
for variant, name in (('findings', 'Findings only'), ('impression', 'Impression only'), ('full', 'both')):
    r = raw_res[variant]
    print(f"{name:14s} {r['val']:7.4f} {r['test']:7.4f} {r['test_hasF']:26.4f}   {lexical[variant]['test']:.4f}")
print()
d = raw_res['full']['test'] - raw_res['findings']['test']
print(f"With the same encoder, dropping the Impression changes the label recovery by {-d:+.4f} test AUROC "
      f"({raw_res['full']['test']:.4f} -> {raw_res['findings']['test']:.4f}).")

del _bert; torch.cuda.empty_cache() if torch.cuda.is_available() else None   # free the GPU before Stage 1

In [ ]:
# Diagnostics verdict -- always printed, even when RUN_STAGE1 = False
print('=== DIAGNOSTICS VERDICT (Sections 4-5) ===')
print(f"label recovery, test macro AUROC      word-level   frozen ClinicalBERT")
for variant, name in (('findings', 'Findings only'), ('impression', 'Impression only'), ('full', 'Findings + Impression')):
    print(f"  {name:22s}  {lexical[variant]['test']:.4f}       {raw_res[variant]['test']:.4f}")
print()
print('For comparison, every earlier image-only result is far lower (Phase 2 converged image probe: 0.6944 test).')
lex_drop = lexical['full']['test'] - lexical['findings']['test']
raw_drop = raw_res['full']['test'] - raw_res['findings']['test']
print(f"Dropping the Impression costs {lex_drop:.4f} AUROC at word level and {raw_drop:.4f} through frozen ClinicalBERT.")
if lexical['findings']['test'] > 0.95 and lex_drop < 0.03:
    print('The label signal lives in the Findings, not the Impression: Findings alone recovers the report-derived labels almost as well as the')
    print('full report at word level. A text-only classifier on Findings-only text can still read the labels from the report itself, so')
    print('report-derived MeSH labels cannot test whether text helps, whichever section is used.')
elif lex_drop >= 0.03:
    print('The Impression carries a material share of the label signal, so Findings-only text may weaken the shortcut -- see the Stage 1 results below.')
else:
    print('Findings alone recovers the labels less well than the full report; see the Stage 1 results below for how much of the shortcut remains.')


## 6. Stage 1: dual-tower contrastive pretraining on `TEXT_VARIANT` text

Same model and recipe as `cxr_recall_phase2_kaggle.ipynb` Section 7 (ViT-B/16 image tower, Bio_ClinicalBERT text tower, 512-d shared space, symmetric InfoNCE, mixed precision, image/projection LR `1e-4`, text LR `2e-5`, early stopping on validation contrastive loss), with two changes: rows with no text for the chosen variant are not used as pairs, and **same-study off-diagonal pairs are masked out of the negatives** (`MASK_SAME_STUDY`).

In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    import math
    import torch.nn.functional as F
    from transformers import AutoModel, AutoTokenizer
    from torch.amp import autocast, GradScaler

    USE_AMP = device.type == 'cuda'
    PID_TO_INT = {p: i for i, p in enumerate(manifest['patient_id'].unique())}

    class IUXrayPairsDataset(Dataset):
        """(image, text, study id as int) -- study ids let the loss mask same-study pairs. Text = manifest['report_text']
        (the TEXT_VARIANT column)."""
        def __init__(self, df, image_root, transform):
            self.df = df.reset_index(drop=True)
            self.image_root = image_root
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
            image = self.transform(image)
            return image, str(row['report_text']), PID_TO_INT[row['patient_id']]


    class TextEncoder(nn.Module):
        def __init__(self, model_name='emilyalsentzer/Bio_ClinicalBERT'):
            super().__init__()
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name)
            self.embed_dim = self.model.config.hidden_size

        def forward(self, texts, device, max_length=192):   # 192 covers 99%+ of reports (earlier token-length check)
            tokens = self.tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
            return self.model(**tokens).last_hidden_state[:, 0, :]


    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=512):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)

        def forward(self, x):
            return F.normalize(self.proj(x), dim=-1)


    class DualTowerContrastiveModel(nn.Module):
        def __init__(self, embed_dim=512, pretrained=True, text_model_name='emilyalsentzer/Bio_ClinicalBERT'):
            super().__init__()
            weights = ViT_B_16_Weights.DEFAULT if pretrained else None
            self.image_backbone = vit_b_16(weights=weights)
            self.image_backbone.heads = nn.Identity()
            self.text_encoder = TextEncoder(text_model_name)
            self.image_proj = ProjectionHead(768, embed_dim)
            self.text_proj = ProjectionHead(self.text_encoder.embed_dim, embed_dim)
            self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

        def encode_image(self, images):
            return self.image_proj(self.image_backbone(images))

        def encode_text(self, texts, device):
            return self.text_proj(self.text_encoder(texts, device))

        def forward(self, images, texts, device):
            return self.encode_image(images), self.encode_text(texts, device)


    def clip_contrastive_loss(image_embeds, text_embeds, logit_scale, study_ids=None):
        """Symmetric InfoNCE. study_ids: (B,) int tensor. When given, every OFF-diagonal pair from the same study is removed
        from the negatives (its logit is set to -1e4): two views of one study share one report, so image A matching text B
        is a correct match, not a negative. Diagonal entries (the true pairs) are never masked."""
        B = image_embeds.shape[0]
        if B < 2:
            raise ValueError(f'clip_contrastive_loss requires batch_size >= 2, got {B}')
        logits = logit_scale.exp() * image_embeds @ text_embeds.t()
        if study_ids is not None:
            same = study_ids[:, None] == study_ids[None, :]
            logits = logits.masked_fill(same & ~torch.eye(B, dtype=torch.bool, device=logits.device), -1e4)
        targets = torch.arange(B, device=image_embeds.device)
        return (F.cross_entropy(logits, targets) + F.cross_entropy(logits.t(), targets)) / 2

    print('Stage 1 model/loss defined')


In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    set_seed(SEED)

    for _stale in ('model_stage1', 'optimizer_s1', 'scaler'):      # free a previous attempt's GPU memory if this cell is re-run
        if _stale in globals():
            del globals()[_stale]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Pairs: only rows that HAVE text for this variant (an empty report has nothing to align with, and identical
    # empty-text embeddings would act as false negatives for each other).
    train_pairs_df, val_pairs_df = train_df[train_df['has_text']], val_df[val_df['has_text']]
    print(f"Stage 1 pairs: train {len(train_pairs_df)}/{len(train_df)}, val {len(val_pairs_df)}/{len(val_df)} "
          f"(excluded: rows with no {TEXT_VARIANT!r} text)")

    train_loader_pairs = DataLoader(IUXrayPairsDataset(train_pairs_df, IMAGE_ROOT, transform), batch_size=STAGE1_BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    val_loader_pairs = DataLoader(IUXrayPairsDataset(val_pairs_df, IMAGE_ROOT, transform), batch_size=STAGE1_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=True)

    model_stage1 = DualTowerContrastiveModel().to(device)
    scaler = GradScaler(device.type, enabled=USE_AMP)

    LR_TEXT = 2e-5
    text_params = list(model_stage1.text_encoder.parameters())
    other_params = [p for p in model_stage1.parameters() if not any(p is tp for tp in text_params)]
    optimizer_s1 = torch.optim.Adam([{'params': other_params, 'lr': LR}, {'params': text_params, 'lr': LR_TEXT}])

    best_val_loss_s1, best_state_s1, epochs_without_improvement_s1 = float('inf'), None, 0
    for epoch in range(1, EPOCHS_REAL + 1):
        model_stage1.train()
        total_loss, train_n = 0.0, 0
        for images, texts, pids in train_loader_pairs:
            images, pids = images.to(device), pids.to(device)
            optimizer_s1.zero_grad()
            with autocast(device.type, enabled=USE_AMP):
                image_emb, text_emb = model_stage1(images, texts, device)
                loss = clip_contrastive_loss(image_emb, text_emb, model_stage1.logit_scale, pids if MASK_SAME_STUDY else None)
            scaler.scale(loss).backward()
            scaler.step(optimizer_s1)
            scaler.update()
            model_stage1.logit_scale.data.clamp_(0, math.log(100))   # CLIP's published bound
            total_loss += loss.item() * images.size(0)
            train_n += images.size(0)
        total_loss /= train_n

        model_stage1.eval()
        val_loss_total, val_n = 0.0, 0
        with torch.no_grad(), autocast(device.type, enabled=USE_AMP):
            for images, texts, pids in val_loader_pairs:
                images, pids = images.to(device), pids.to(device)
                image_emb, text_emb = model_stage1(images, texts, device)
                val_loss_total += clip_contrastive_loss(image_emb, text_emb, model_stage1.logit_scale, pids if MASK_SAME_STUDY else None).item() * images.size(0)
                val_n += images.size(0)
        val_loss = val_loss_total / val_n
        print(f'[Stage 1 | {TEXT_VARIANT}] epoch {epoch}/{EPOCHS_REAL}  train_loss={total_loss:.4f}  val_loss={val_loss:.4f}')

        if val_loss < best_val_loss_s1:
            best_val_loss_s1, best_state_s1, epochs_without_improvement_s1 = val_loss, copy.deepcopy(model_stage1.state_dict()), 0
        else:
            epochs_without_improvement_s1 += 1
            if epochs_without_improvement_s1 >= PATIENCE:
                print(f'no val improvement for {PATIENCE} epochs -- stopping early at epoch {epoch}')
                break

    model_stage1.load_state_dict(best_state_s1)
    print(f'best Stage 1 val_loss = {best_val_loss_s1:.4f}')
    torch.save({'model_state_dict': model_stage1.state_dict(), 'text_variant': TEXT_VARIANT}, f'/kaggle/working/stage1_{TEXT_VARIANT}.pt')


## 7. Cache frozen embeddings (all rows, including rows with empty text) + alignment diagnostic

One pass extracts both towers' embeddings for train/val/test; everything below runs on these arrays. Saved to `/kaggle/working/phase2_findings_embeddings_<variant>.npz`. The recall@1 diagnostic uses **one view per study and only rows with text** (so duplicate reports cannot make the target ambiguous), against a random baseline of 1/n.

In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    class IUXrayMultimodalDatasetP2(Dataset):
        """(image, text, label) for caching both modalities' embeddings against the real labels; text = the TEXT_VARIANT column."""
        def __init__(self, df, image_root, transform, label_vocab):
            self.df = df.reset_index(drop=True)
            self.image_root = image_root
            self.transform = transform
            self.label_vocab = label_vocab

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
            image = self.transform(image)
            tags = row['mesh_tags'].split(';') if row['mesh_tags'] else []
            return image, str(row['report_text']), mesh_tags_to_vector(tags, self.label_vocab)

    for p in model_stage1.parameters():
        p.requires_grad = False
    model_stage1.eval()

    @torch.no_grad()
    def extract_dual_embeddings(model, loader, device):
        model.eval()
        img_embs, txt_embs, labels_out = [], [], []
        with autocast(device.type, enabled=USE_AMP):
            for images, texts, labels in loader:
                img_embs.append(model.encode_image(images.to(device)).float().cpu().numpy())
                txt_embs.append(model.encode_text(texts, device).float().cpu().numpy())
                labels_out.append(labels.numpy())
        return np.concatenate(img_embs), np.concatenate(txt_embs), np.concatenate(labels_out)

    emb = {}
    for name, df in (('train', train_df), ('val', val_df), ('test', test_df)):
        loader = DataLoader(IUXrayMultimodalDatasetP2(df, IMAGE_ROOT, transform, label_vocab), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
        emb[name] = extract_dual_embeddings(model_stage1, loader, device)
    tr_img, tr_txt, tr_lbl = emb['train']; va_img, va_txt, va_lbl = emb['val']; te_img, te_txt, te_lbl = emb['test']
    assert np.array_equal(tr_lbl, y_tr) and np.array_equal(va_lbl, y_va) and np.array_equal(te_lbl, y_te)   # row order == split order
    print('cached shapes:', tr_img.shape, tr_txt.shape, va_img.shape, te_img.shape)
    np.savez_compressed(f'/kaggle/working/phase2_findings_embeddings_{TEXT_VARIANT}.npz', train_img=tr_img, train_txt=tr_txt, train_lbl=tr_lbl,
                        val_img=va_img, val_txt=va_txt, val_lbl=va_lbl, test_img=te_img, test_txt=te_txt, test_lbl=te_lbl,
                        text_variant=TEXT_VARIANT, stage1_batch_size=STAGE1_BATCH_SIZE, label_vocab=np.array(label_vocab))

    # alignment diagnostic on one view per study, rows with text only
    seen, keep = set(), []
    for i, (p, h) in enumerate(zip(val_df['patient_id'], val_df['has_text'])):
        if h and p not in seen:
            seen.add(p); keep.append(i)
    sims = va_img[keep] @ va_txt[keep].T
    n_diag = len(keep)
    i2t = float((sims.argmax(1) == np.arange(n_diag)).mean()); t2i = float((sims.argmax(0) == np.arange(n_diag)).mean())
    print(f'[Stage 1 diagnostic | {TEXT_VARIANT}] image->text recall@1 = {i2t:.4f}  text->image recall@1 = {t2i:.4f}  (n={n_diag} studies, random = {1 / n_diag:.4f})')


## 8. Converged probes: image-only, text-only, fusion (and a gated head)

Linear heads on the frozen embeddings, trained to convergence: sklearn L-BFGS logistic regression for the four convex probes (`image_only`, `text_only`, `full_sum` = the ungated fusion head, `full_concat` = a more flexible fusion), and the torch gated head at lr `1e-2` with a patience-50 stop that reports whether it actually stopped. Scored on all rows, on rows that have text, and on rows whose text is **not** an exact duplicate of any training report. The text-minus-image gap on each subset is the circularity read-out.

In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    feat = {
        'image_only':  (tr_img, va_img, te_img),
        'text_only':   (tr_txt, va_txt, te_txt),
        'full_sum':    (tr_img + tr_txt, va_img + va_txt, te_img + te_txt),
        'full_concat': (np.hstack([tr_img, tr_txt]), np.hstack([va_img, va_txt]), np.hstack([te_img, te_txt])),
    }
    probe = {}
    for name, (Xtr, Xva, Xte) in feat.items():
        C, pv, pt = fit_linear_probe(Xtr, y_tr, Xva, y_va, Xte, Cs=(0.3, 1, 3, 10, 30, 100, 300, 1000))
        probe[name] = {'C': C, 'pv': pv, 'pt': pt}

    # gated fusion head (torch), trained at lr 1e-2 with early stopping
    class GatedProbe(nn.Module):
        def __init__(self, d, k):
            super().__init__()
            self.head = nn.Linear(d, k)
            self.gate = nn.Linear(2 * d, 1)
        def forward(self, i, t):
            g = torch.sigmoid(self.gate(torch.cat([i, t], -1)))
            return self.head(g * i + (1 - g) * t)

    set_seed(SEED)
    D = tr_img.shape[1]
    gm = GatedProbe(D, len(label_vocab)).to(device)
    g_opt = torch.optim.Adam(gm.parameters(), lr=1e-2)
    TI, TT, TL = (torch.tensor(a).to(device) for a in (tr_img, tr_txt, y_tr))
    VI, VT = torch.tensor(va_img).to(device), torch.tensor(va_txt).to(device)
    EI, ET = torch.tensor(te_img).to(device), torch.tensor(te_txt).to(device)
    g_best, g_state, g_bad, g_stopped, G_MAX, G_PAT = -1.0, None, 0, False, 3000, 50
    for ep in range(1, G_MAX + 1):
        gm.train()
        perm = torch.randperm(len(TL), device=device)
        for s in range(0, len(perm), 64):
            idx = perm[s:s + 64]
            g_opt.zero_grad(); criterion(gm(TI[idx], TT[idx]), TL[idx]).backward(); g_opt.step()
        gm.eval()
        with torch.no_grad():
            v = macro_auroc(y_va, torch.sigmoid(gm(VI, VT)).cpu().numpy())
        if v > g_best:
            g_best, g_state, g_bad, g_best_ep = v, copy.deepcopy(gm.state_dict()), 0, ep
        else:
            g_bad += 1
            if g_bad >= G_PAT:
                g_stopped = True
                break
    gm.load_state_dict(g_state); gm.eval()
    with torch.no_grad():
        probe['gated_full'] = {'C': None, 'pv': torch.sigmoid(gm(VI, VT)).cpu().numpy(), 'pt': torch.sigmoid(gm(EI, ET)).cpu().numpy()}
    print(f"gated head: best val {g_best:.4f} at epoch {g_best_ep}; " + (f'early-stopped at epoch {ep} (converged)' if g_stopped else f'HIT THE {G_MAX}-EPOCH CAP (NOT converged -- number is a lower bound)'))

    # subsets of the test set
    train_texts = set(train_df.loc[train_df['has_text'], 'report_text'])
    has_text_te = test_df['has_text'].to_numpy()
    nondup_te = (test_df['has_text'] & ~test_df['report_text'].isin(train_texts)).to_numpy()
    print(f"test rows: all {len(test_df)}, with text {int(has_text_te.sum())}, with text NOT duplicated in train {int(nondup_te.sum())}")

    REF = {'image_only': (0.7078, 0.6944), 'text_only': (0.9423, 0.9355), 'full_sum': (0.9395, 0.9329), 'full_concat': (0.9428, 0.9311), 'gated_full': (0.9440, 0.9404)}
    print(f"\n=== Stage 2 probes, TEXT_VARIANT={TEXT_VARIANT!r} (macro AUROC) ===")
    print(f"{'probe':12s} {'val':>7s} {'TEST all':>9s} {'TEST w/ text':>13s} {'TEST non-dup':>13s}   full-text reference (val / TEST; a DIFFERENT Stage 1 run)")
    for name in ('image_only', 'text_only', 'full_sum', 'full_concat', 'gated_full'):
        p = probe[name]
        rv, rt = REF[name]
        print(f"{name:12s} {masked_auroc(y_va, p['pv']):7.4f} {masked_auroc(y_te, p['pt']):9.4f} {masked_auroc(y_te, p['pt'], has_text_te):13.4f} "
              f"{masked_auroc(y_te, p['pt'], nondup_te):13.4f}   {rv:.4f} / {rt:.4f}")
    print()
    for label, mask in (('all rows', None), ('rows with text', has_text_te), ('rows with non-duplicate text', nondup_te)):
        gap = masked_auroc(y_te, probe['text_only']['pt'], mask) - masked_auroc(y_te, probe['image_only']['pt'], mask)
        print(f"text-only minus image-only (TEST, {label}): {gap:+.4f}")
    fus = masked_auroc(y_te, probe['full_sum']['pt']) - masked_auroc(y_te, probe['text_only']['pt'])
    print(f"full_sum minus text_only (TEST, all rows): {fus:+.4f}   <- does the image add anything once text is present?")


## 9. Arm 2 (retrieval quality) on this Stage 1's image embeddings

In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    index_f = build_index(tr_img)
    _, va_nb = search(index_f, va_img, K)
    _, te_nb = search(index_f, te_img, K)
    ag = lambda ql, nb, pool_l: float(np.mean([label_agreement_at_k(ql[i], pool_l[nb[i]]) for i in range(len(ql))]))
    mr = lambda ql, nb, pool_l: float(np.mean([mean_reciprocal_rank(ql[i], pool_l[nb[i]]) for i in range(len(ql))]))
    arm2 = {'val_agree': ag(y_va, va_nb, y_tr), 'test_agree': ag(y_te, te_nb, y_tr), 'val_mrr': mr(y_va, va_nb, y_tr), 'test_mrr': mr(y_te, te_nb, y_tr)}
    print(f"[Arm 2 | {TEXT_VARIANT}] label_agreement@{K}  val {arm2['val_agree']:.4f}  TEST {arm2['test_agree']:.4f}   MRR val {arm2['val_mrr']:.4f}  TEST {arm2['test_mrr']:.4f}")
    print('references, TEST label_agreement@5: Phase 1 audit rerun 0.2864; Phase 2 full-text run B 0.2732; Phase 2 run A 0.2428; random 0.1845')


## 10. What this run says

In [ ]:
if RUN_STAGE1:  # set RUN_STAGE1 = False in the configuration cell to run only the cheap diagnostics
    txt_all = masked_auroc(y_te, probe['text_only']['pt'])
    img_all = masked_auroc(y_te, probe['image_only']['pt'])
    print(f"=== SUMMARY (TEXT_VARIANT = {TEXT_VARIANT!r}, test macro AUROC) ===")
    print(f"word-level ceiling from this text          : {lexical[TEXT_VARIANT]['test']:.4f}")
    print(f"frozen pretrained ClinicalBERT probe       : {raw_res[TEXT_VARIANT]['test']:.4f}")
    print(f"Stage 1 text-only probe                    : {txt_all:.4f}   (full-text reference, a different Stage 1 run: {REF['text_only'][1]:.4f})")
    print(f"Stage 1 image-only probe                   : {img_all:.4f}   (full-text reference: {REF['image_only'][1]:.4f})")
    print(f"text - image gap                           : {txt_all - img_all:+.4f}   (full-text reference: {REF['text_only'][1] - REF['image_only'][1]:+.4f})")
    print(f"fusion minus text-only                     : {masked_auroc(y_te, probe['full_sum']['pt']) - txt_all:+.4f}")
    print()
    if txt_all > 0.9:
        print('The text-only probe still scores above 0.90. Dropping or keeping the Impression does not change the conclusion: with report-derived')
        print('MeSH labels the text modality recovers the labels from the report itself, and no section choice turns this into a test of whether text helps.')
    elif txt_all > img_all + 0.05:
        print('Text-only is lower than with the full report but still far above the image: part of the shortcut remains.')
    else:
        print('The text advantage is largely gone for this text variant -- examine Section 4 per-label results to see which labels carried it.')
    print('Reminder: Stage 1 here differs from the earlier full-text runs (same-study masking, empty-text rows dropped, run-to-run noise ~0.01 on probes);')
    print("for an apples-to-apples Stage 1 comparison rerun with TEXT_VARIANT = 'full'.")
